# RavenStack Customer & Revenue Analysis
### What Drives Customer Retention and Revenue Expansion at RavenStack?

**Author:** Elene Kikalishvili  
**Dataset:** SaaS Subscription & Churn Analytics Dataset by River @ Rivalytics  
**Tools:** Python · pandas · matplotlib · seaborn

---
## Project Overview

RavenStack is a stealth-mode B2B SaaS startup delivering AI-powered team collaboration and developer tools. After a closed pilot with coding bootcamp graduates, the company is preparing for public launch across three subscription tiers - Basic, Pro, and Enterprise.

Despite steady account growth, the business faces elevated churn and uneven revenue expansion. This analysis investigates what drives long-term customer retention and revenue growth using five interconnected data tables spanning subscriptions, product usage, support activity, and churn events.

**Key Questions:**
- Q1: Which customer segments generate the highest long-term revenue, and which expand their contracts over time?
- Q2: Which referral sources and industries convert trials to paid subscriptions best?
- Q3: What does the journey from trial → upgrade → churn look like, and where does revenue expand or collapse?
- Q4: Which features drive product value, and do engaged users retain better?
- Q5: Which support experience patterns are associated with increased churn risk?
- Q6: How do different signup cohorts retain and expand over time?

---
## Section 0 - Data Understanding & Preparation
### 0.1 Imports & Loading
### 0.2 Exploration
### 0.3 Cleaning
### 0.4 Merge Strategy
### 0.5 Data Quality & Assumptions
### 0.6 Metric Definitions & Business Logic

---
## Section 1 - Executive KPI Snapshot

---
## Q1 - Revenue & Expansion
Which customer segments generate the highest long-term revenue, and which expand their contracts over time?

---
## Q2 - Trial Conversion
Which referral sources and industries convert trials to paid subscriptions best?

---
## Q3 - Customer Lifecycle
What does the journey from trial → upgrade → churn look like, and where does revenue expand or collapse along the way?

---
## Q4 - Feature Adoption & Engagement
Which features drive product value, and do engaged users retain better?

---
## Q5 - Support Quality
Which support experience patterns are associated with increased churn risk?

---
## Q6 - Cohort Analysis
How do different signup cohorts retain and expand over time?

---
## Section 7 - Key Analytical Findings

---
## Section 8 - Recommendations

---
## Project Overview

RavenStack is a stealth-mode B2B SaaS startup delivering AI-powered team collaboration and developer tools. After a closed pilot with coding bootcamp graduates, the company is preparing for public launch across three subscription tiers - Basic, Pro, and Enterprise.

Despite steady account growth, the business faces elevated churn and uneven revenue expansion. This analysis investigates what drives long-term customer retention and revenue growth using five interconnected data tables spanning subscriptions, product usage, support activity, and churn events.

**Key Questions:**
- Q1: Which customer segments generate the highest long-term revenue, and which expand their contracts over time?
- Q2: Which referral sources and industries convert trials to paid subscriptions best?
- Q3: What does the journey from trial → upgrade → churn look like, and where does revenue expand or collapse?
- Q4: Which features drive product value, and do engaged users retain better?
- Q5: Which support experience patterns are associated with increased churn risk?
- Q6: How do different signup cohorts retain and expand over time?

---

## Section 0 - Data Understanding & Preparation
### 0.1 Imports & Loading

In [56]:
# 0.1 - Imports & Loading 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load all five tables

accounts = pd.read_csv('data/ravenstack_accounts.csv')
subscriptions = pd.read_csv('data/ravenstack_subscriptions.csv')
feature_usage = pd.read_csv('data/ravenstack_feature_usage.csv')
support_tickets = pd.read_csv('data/ravenstack_support_tickets.csv')
churn_events = pd.read_csv('data/ravenstack_churn_events.csv')

print("All tables loaded successfully")
print(f"accounts: {accounts.shape}")
print(f"subscriptions: {subscriptions.shape}")
print(f"feature_usage: {feature_usage.shape}")
print(f"support_tickets: {support_tickets.shape}")
print(f"churn_events: {churn_events.shape}")


All tables loaded successfully
accounts: (500, 10)
subscriptions: (5000, 14)
feature_usage: (25000, 8)
support_tickets: (2000, 9)
churn_events: (600, 9)


### 0.2 Exploration
#### accounts

In [32]:
# accounts table - structure and first rows

print("ACCOUNTS TABLE")
print(f"shape: {accounts.shape}")
print("\n--- First 5 rows ---")
print(accounts.head())
print("\n--- Column Types & Nulls ---")
print(accounts.info())


ACCOUNTS TABLE
shape: (500, 10)

--- First 5 rows ---
  account_id account_name    industry country signup_date referral_source  \
0   A-2e4581    Company_0      EdTech      US  2024-10-16         partner   
1   A-43a9e3    Company_1     FinTech      IN  2023-08-17           other   
2   A-0a282f    Company_2    DevTools      US  2024-08-27         organic   
3   A-1f0ac7    Company_3  HealthTech      UK  2023-08-27           other   
4   A-ce550d    Company_4  HealthTech      US  2024-10-27           event   

    plan_tier  seats  is_trial  churn_flag  
0       Basic      9     False       False  
1       Basic     18     False        True  
2       Basic      1     False       False  
3       Basic     24      True       False  
4  Enterprise     35     False        True  

--- Column Types & Nulls ---
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 

**Structure:**  
The accounts table has 500 rows and 10 columns with no nulls across any column - the table is complete.  
Column types are mostly strings, with `seats` as integer and `is_trial` and `churn_flag` as booleans.  
One issue to fix in cleaning: `signup_date` is currently a string and needs to be converted to a proper datetime type for cohort analysis in Q6.

In [11]:
# numeric stats

print("--- Numeric Column Stats ---")
print(accounts.describe())


--- Numeric Column Stats ---
            seats
count  500.000000
mean    20.560000
std     21.044718
min      1.000000
25%      5.000000
50%     15.000000
75%     28.000000
max    163.000000


**Seats distribution:**  
The median account has 15 licensed seats, suggesting most RavenStack customers are small to mid-sized teams.  
The mean (20.5) is higher than the median, suggesting the distribution is right-skewed - some accounts have significantly more seats than the typical customer, with the largest account holding 163 seats.

In [18]:
# categorical distributions and checks

print("--- Plan Tier Distribution ---")
print(accounts['plan_tier'].value_counts())

print("\n--- Referral Source Distribution ---")
print(accounts['referral_source'].value_counts())

print("\n--- Industry Distribution ---")
print(accounts['industry'].value_counts())

print("\n--- Trial vs Non-Trial ---")
print(accounts['is_trial'].value_counts())

print("\n--- Churn Flag Distribution ---")
print(accounts['churn_flag'].value_counts())

print("\n--- Duplicate account_id check ---")
print(f"{accounts['account_id'].duplicated().sum()} duplicate account_ids")


--- Plan Tier Distribution ---
plan_tier
Pro           178
Basic         168
Enterprise    154
Name: count, dtype: int64

--- Referral Source Distribution ---
referral_source
organic    114
other      103
ads         98
event       96
partner     89
Name: count, dtype: int64

--- Industry Distribution ---
industry
DevTools         113
FinTech          112
Cybersecurity    100
HealthTech        96
EdTech            79
Name: count, dtype: int64

--- Trial vs Non-Trial ---
is_trial
False    403
True      97
Name: count, dtype: int64

--- Churn Flag Distribution ---
churn_flag
False    390
True     110
Name: count, dtype: int64

--- Duplicate account_id check ---
0 duplicate account_ids


**Distributions & checks:**  

**Plan tier** - fairly evenly distributed: Pro (178), Basic (168), Enterprise (154). No single tier dominates.

**Referral source** - organic leads (114), followed closely by other, ads, event, and partner. Fairly balanced, no single channel overwhelmingly dominates.

**Industry** - DevTools (113) and FinTech (112) are the largest segments, EdTech (79) the smallest. Five industries total, clean categories, no typos.

**Trial vs non-trial** - 97 accounts are currently on trial, 403 are not. That's about 19% of accounts in trial status.

**Churn flag** - 110 accounts have churned (True), 390 are active (False).

**Duplicates** - 0 duplicate account_ids. The table is clean.

---
#### subscriptions

In [30]:
# subscriptions - structure and first rows 

print("SUBSCRIPTIONS TABLE")
print(f"Shape: {subscriptions.shape}")
print("\n--- First 5 rows ---")
print(subscriptions.head())
print("\n--- Column Types & Nulls ---")
print(subscriptions.info())


SUBSCRIPTIONS TABLE
Shape: (5000, 14)

--- First 5 rows ---
  subscription_id account_id  start_date    end_date   plan_tier  seats  \
0        S-8cec59   A-3c1a3f  2023-12-23  2024-04-12  Enterprise     14   
1        S-0f6f44   A-9b9fe9  2024-06-11         NaN         Pro     17   
2        S-51c0d1   A-659280  2024-11-25         NaN  Enterprise     62   
3        S-f81687   A-e7a1e2  2024-11-23  2024-12-13  Enterprise      5   
4        S-cff5a2   A-ba6516  2024-01-10         NaN  Enterprise     27   

   mrr_amount  arr_amount  is_trial  upgrade_flag  downgrade_flag  churn_flag  \
0        2786       33432     False         False           False        True   
1         833        9996     False         False           False       False   
2           0           0      True          True           False       False   
3         995       11940     False         False           False        True   
4        5373       64476     False         False           False       False   

  

**Structure:**  
The subscriptions table has 5,000 rows and 14 columns across 500 accounts - an average of 10 subscription records per account, reflecting trials, upgrades, downgrades, and renewals over time.  
No nulls except `end_date`, which has only 486 non-null values - this is expected, as null `end_date` indicates an active subscription.  
Both `start_date` and `end_date` are currently strings and need to be converted to datetime in cleaning.  
One observation: trial subscriptions (row 2 in head) show `mrr_amount = 0` - trials will need to be excluded when calculating active MRR.  

In [34]:
# numeric stats

print("--- Numeric Column Stats ---")
print(subscriptions.describe())


--- Numeric Column Stats ---
             seats    mrr_amount     arr_amount
count  5000.000000   5000.000000    5000.000000
mean     29.852000   2267.749400   27212.992800
std      23.089771   3421.375348   41056.504178
min       1.000000      0.000000       0.000000
25%      14.000000    285.000000    3420.000000
50%      24.000000    931.000000   11172.000000
75%      40.000000   2786.000000   33432.000000
max     189.000000  33830.000000  405960.000000


**Numeric stats:**  
Median MRR per subscription is \\$931, but the mean is \\$2,268 - a significant gap indicating a right-skewed distribution where a small number of high-value subscriptions pull the average up considerably.  
The minimum MRR of \\$0 confirms trial subscriptions are included in this table and must be excluded from active MRR calculations.  
The largest subscription generates \\$33,830 MRR (\\$405,960 ARR), suggesting at least one significant enterprise contract in the dataset.


In [27]:
# categorical distributions and checks

print("--- Plan Tier Distribution ---")
print(subscriptions['plan_tier'].value_counts())

print("\n--- Billing Frequency ---")
print(subscriptions['billing_frequency'].value_counts())

print("\n--- Trial Subscriptions ---")
print(subscriptions['is_trial'].value_counts())

print("\n--- Upgrade / Downgrade Flags ---")
print(f"Upgrades:   {subscriptions['upgrade_flag'].sum()}")
print(f"Downgrades: {subscriptions['downgrade_flag'].sum()}")

print("\n--- Duplicate subscription_id check ---")
print(f"Duplicate subscription_ids: {subscriptions['subscription_id'].duplicated().sum()}")


--- Plan Tier Distribution ---
plan_tier
Enterprise    1723
Pro           1675
Basic         1602
Name: count, dtype: int64

--- Billing Frequency ---
billing_frequency
monthly    2539
annual     2461
Name: count, dtype: int64

--- Trial Subscriptions ---
is_trial
False    4222
True      778
Name: count, dtype: int64

--- Upgrade / Downgrade Flags ---
Upgrades:   529
Downgrades: 218

--- Duplicate subscription_id check ---
Duplicate subscription_ids: 0


**Distributions & checks:**  

**Plan tier** - Enterprise has the most subscription records (1,723), followed by Pro (1,675) and Basic (1,602). Fairly balanced but Enterprise slightly leads - important because Enterprise subscriptions carry the highest MRR.  

**Billing frequency** - almost a 50/50 split between monthly (2,539) and annual (2,461). Annual subscribers typically have higher LTV and lower churn risk.  

**Trial subscriptions** - 778 trial records out of 5,000 total. These will be excluded from active MRR calculations.  

**Upgrades vs downgrades** - 529 upgrades and 218 downgrades. Upgrades outnumber downgrades by more than 2:1, which is a positive signal for revenue expansion.   

**Duplicates** - 0 duplicate subscription_ids. Clean.

---
#### feature_usage

In [31]:
# feature_usage - structure and first rows

print("FEATURE USAGE TABLE")
print(f"Shape: {feature_usage.shape}")
print("\n--- First 5 rows ---")
print(feature_usage.head())
print("\n--- Column Types & Nulls ---")
print(feature_usage.info())


FEATURE USAGE TABLE
Shape: (25000, 8)

--- First 5 rows ---
   usage_id subscription_id  usage_date feature_name  usage_count  \
0  U-1c6c24        S-0fcf7d  2023-07-27   feature_20            9   
1  U-f07cb8        S-c25263  2023-08-07    feature_5            9   
2  U-096807        S-f29e7f  2023-12-07    feature_3            9   
3  U-6b1580        S-be655e  2024-07-28   feature_40            5   
4  U-720a29        S-f9b1d0  2024-12-02   feature_12           12   

   usage_duration_secs  error_count  is_beta_feature  
0                 5004            0            False  
1                  369            0            False  
2                 1458            0            False  
3                 2085            0            False  
4                  900            0            False  

--- Column Types & Nulls ---
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype
---  ------         

**Structure:**  
The feature_usage table has 25,000 rows and 8 columns with no nulls across any column - the table is complete.  
This is the largest table in the dataset and links to subscriptions via `subscription_id`.   
`usage_date` is currently a string and needs to be converted to datetime in cleaning.  
The three numeric columns - `usage_count`,`usage_duration_secs`, and `error_count` - are the inputs for the engagement score calculated in Q4.

In [35]:
# numeric stats

print("--- Numeric Column Stats ---")
print(feature_usage.describe())


--- Numeric Column Stats ---
        usage_count  usage_duration_secs   error_count
count  25000.000000         25000.000000  25000.000000
mean      10.021000          3042.202880      0.564280
std        3.143729          2056.544615      1.012595
min        0.000000             0.000000      0.000000
25%        8.000000          1350.000000      0.000000
50%       10.000000          2760.000000      0.000000
75%       12.000000          4400.000000      1.000000
max       26.000000         12696.000000      8.000000


**Numeric stats:**  
Average usage count per event is 10, with a tight distribution (min 0, max 26) - usage frequency is fairly consistent across records.  
Usage duration varies more widely - median is 2,760 seconds (~46 minutes) but the mean is 3,042 seconds (~51 minutes), with a max of 12,696 seconds (~3.5 hours), suggesting some features drive significantly longer sessions.  
Error count is low overall - median is 0 and mean is 0.56, meaning most usage events produce no errors, though the max of 8 errors in a single event is worth monitoring in Q4 feature analysis.

In [39]:
# categorical distributions and checks

print("--- Number of Features ---")
print(f"Unique features: {feature_usage['feature_name'].nunique()}")

print("--- Top 10 Features by Usage Records ---")
print(feature_usage['feature_name'].value_counts().head(10))

print("\n--- Beta vs Standard Features ---")
print(feature_usage['is_beta_feature'].value_counts())

print("\n--- Usage Date Range ---")
print(f"Earliest date: {feature_usage['usage_date'].min()}")
print(f"Latest date:   {feature_usage['usage_date'].max()}")

print("\n--- Subscriptions represented in feature_usage ---")
print(f"Unique subscription_ids: {feature_usage['subscription_id'].nunique()}")
print(f"Total subscriptions in subscriptions table: {subscriptions.shape[0]}")

print("\n--- Duplicate usage_id check ---")
print(f"Duplicate usage_ids: {feature_usage['usage_id'].duplicated().sum()}")


--- Number of Features ---
Unique features: 40
--- Top 10 Features by Usage Records ---
feature_name
feature_12    659
feature_32    659
feature_6     655
feature_17    651
feature_34    650
feature_26    649
feature_36    648
feature_31    644
feature_20    643
feature_24    643
Name: count, dtype: int64

--- Beta vs Standard Features ---
is_beta_feature
False    22456
True      2544
Name: count, dtype: int64

--- Usage Date Range ---
Earliest date: 2023-01-01
Latest date:   2024-12-31

--- Subscriptions represented in feature_usage ---
Unique subscription_ids: 4967
Total subscriptions in subscriptions table: 5000

--- Duplicate usage_id check ---
Duplicate usage_ids: 21


In [37]:
# investigate duplicate usage_ids 

duplicates = feature_usage[feature_usage['usage_id'].duplicated(keep=False)]
print(f"Total rows involved in duplicates: {len(duplicates)}")
print(duplicates.sort_values('usage_id').head(10))


Total rows involved in duplicates: 42
       usage_id subscription_id  usage_date feature_name  usage_count  \
19294  U-0c9318        S-01b2dc  2023-12-11    feature_9            5   
17533  U-0c9318        S-0ffab0  2024-01-30   feature_11            3   
20588  U-13ce5b        S-9b623b  2023-03-26   feature_28           10   
9626   U-13ce5b        S-8b0950  2024-09-10    feature_9            8   
18480  U-2103bb        S-7fc49b  2023-04-18   feature_12            9   
10379  U-2103bb        S-ae3270  2024-01-06    feature_3           10   
22     U-25b56c        S-810c27  2024-10-06   feature_20            7   
7574   U-25b56c        S-34253c  2023-10-28   feature_20            6   
21376  U-48a4aa        S-93f835  2024-08-24   feature_39           10   
1085   U-48a4aa        S-383ac2  2023-02-12   feature_28           14   

       usage_duration_secs  error_count  is_beta_feature  
19294                 1060            3            False  
17533                 1614            0 

**Distributions & checks:**  
The top 10 features out of 40 by record count show very even distribution, ranging from 643 to 659 records each - no single feature dominates, suggesting analysis of total usage volume and duration will be more meaningful than record count alone.  
Beta features account for approximately 10% of all usage records, matching expected distribution.  
Data spans exactly two years (2023–2024).  
33 subscriptions have no feature usage records - likely new or immediately churned accounts.  
21 duplicate usage_ids were found and investigated - these are ID generation collisions, not true duplicate events, as all other fields differ between pairs.  
All rows will be retained in cleaning.

---
#### support_tickets

In [40]:
# support_tickets - structure and first rows

print("SUPPORT TICKETS TABLE")
print(f"Shape: {support_tickets.shape}")
print("\n--- First 5 rows ---")
print(support_tickets.head())
print("\n--- Column Types & Nulls ---")
support_tickets.info()


SUPPORT TICKETS TABLE
Shape: (2000, 9)

--- First 5 rows ---
  ticket_id account_id submitted_at            closed_at  \
0  T-0024de   A-712f1c   2023-07-27  2023-07-28 03:00:00   
1  T-4d04b9   A-e43bf7   2024-07-08  2024-07-09 03:00:00   
2  T-d5e12f   A-0f3e88   2024-10-17  2024-10-17 19:00:00   
3  T-dfce9a   A-4c56c9   2024-09-08  2024-09-09 23:00:00   
4  T-c59f77   A-6f8ad2   2024-11-30  2024-12-01 02:00:00   

   resolution_time_hours priority  first_response_time_minutes  \
0                   27.0     high                           74   
1                   27.0   urgent                          144   
2                   19.0   urgent                           93   
3                   47.0   medium                          126   
4                   26.0   medium                            8   

   satisfaction_score  escalation_flag  
0                 NaN            False  
1                 NaN            False  
2                 4.0            False  
3                

**Structure:**  
The support_tickets table has 2,000 rows and 9 columns, linking to accounts via `account_id`.  
The only null column is `satisfaction_score` with 1,175 non-null values out of 2,000 - meaning 825 tickets (41%) have no satisfaction score, will be excluded from average score calculations.  
Both `submitted_at` and `closed_at` are currently strings and need to be converted to datetime in cleaning.  
All other columns are complete with no nulls.

In [41]:
# numeric stats

print("--- Numeric Column Stats ---")
print(support_tickets.describe())


--- Numeric Column Stats ---
       resolution_time_hours  first_response_time_minutes  satisfaction_score
count            2000.000000                  2000.000000         1175.000000
mean               35.861000                    88.480000            3.981277
std                21.138427                    51.531877            0.809646
min                 1.000000                     1.000000            3.000000
25%                17.000000                    43.000000            3.000000
50%                35.000000                    88.000000            4.000000
75%                54.000000                   131.000000            5.000000
max                72.000000                   180.000000            5.000000


**Numeric stats:**  
Average resolution time is 35.9 hours (median 35 hours), ranging from 1 to 72 hours - suggesting most tickets are resolved within 1-3 days.  
First response time averages 88 minutes (median 88 minutes) with a wide range of 1 to 180 minutes, indicating inconsistent responsiveness across tickets.  
Satisfaction score averages 3.98 out of 5 (median 4.0) across the 1,175 tickets that received a response - a relatively positive signal, though the 825 missing scores mean this average represents only 59% of all tickets.  

In [42]:
# categorical distributions and checks 

print("\n--- Priority Distribution ---")
print(support_tickets['priority'].value_counts())

print("\n--- Escalation Flag ---")
print(support_tickets['escalation_flag'].value_counts())

print("\n--- Accounts represented in support_tickets ---")
print(f"Unique account_ids: {support_tickets['account_id'].nunique()}")

print("\n--- Duplicate ticket_id check ---")
print(f"Duplicate ticket_ids: {support_tickets['ticket_id'].duplicated().sum()}")



--- Priority Distribution ---
priority
urgent    514
high      510
medium    491
low       485
Name: count, dtype: int64

--- Escalation Flag ---
escalation_flag
False    1905
True       95
Name: count, dtype: int64

--- Accounts represented in support_tickets ---
Unique account_ids: 492

--- Duplicate ticket_id check ---
Duplicate ticket_ids: 0


**Distributions & checks:**  
Priority is evenly distributed across all four levels - urgent (514), high (510), medium (491), low (485) - no single priority dominates, suggesting the dataset is synthetically balanced rather than reflecting a real-world skew toward lower priority tickets.  
Escalation is rare - only 95 out of 2,000 tickets (4.75%) were escalated. This is a small but analytically important segment for Q5, as escalated tickets likely correlate with dissatisfied customers.  
492 out of 500 accounts appear in support_tickets - 8 accounts never submitted a ticket. This is a minor gap with no impact on analysis.  
No duplicate ticket_ids - the table is clean.

---
#### churn_events

In [49]:
# churn_events - structure and first rows

print("CHURN EVENTS TABLE")
print(f"Shape: {churn_events.shape}")
print("\n--- First 5 rows ---")
print(churn_events.head())
print("\n--- Column Types & Nulls ---")
churn_events.info()


CHURN EVENTS TABLE
Shape: (600, 9)

--- First 5 rows ---
  churn_event_id account_id  churn_date reason_code  refund_amount_usd  \
0       C-816288   A-c37cab  2024-10-27     pricing               4.03   
1       C-5a81e7   A-37f969  2024-06-25     support              96.45   
2       C-a174be   A-b07346  2024-11-12      budget               0.00   
3       C-accb39   A-1e50e0  2023-11-01      budget              54.94   
4       C-92f889   A-956988  2024-12-30     unknown               0.00   

   preceding_upgrade_flag  preceding_downgrade_flag  is_reactivation  \
0                   False                     False            False   
1                    True                     False            False   
2                   False                     False            False   
3                   False                     False            False   
4                   False                      True             True   

            feedback_text  
0  switched to competitor  
1        

**Structure:**  
The churn_events table has 600 rows and 9 columns, linking to accounts via `account_id`.  
This table records every churn event - one row per churned account.  
`churn_date` is currently a string and needs to be converted to datetime in cleaning.  
The only incomplete column is `feedback_text` with 452 non-null values out of 600 - 148 accounts (25%) left no written feedback, which is expected as free-text responses are optional.  
`refund_amount_usd` has no nulls - confirmed clean, with zero values representing no refund.  
All boolean columns are complete.

In [50]:
# numeric stats

print("--- Numeric Column Stats ---")
print(churn_events.describe()) 


--- Numeric Column Stats ---
       refund_amount_usd
count         600.000000
mean           14.420417
std            39.224591
min             0.000000
25%             0.000000
50%             0.000000
75%             0.000000
max           392.920000


**Numeric stats:**  
The only numeric column is `refund_amount_usd`.  
The median refund is USD 0 across all quartiles up to the 75th percentile - meaning at least 75% of churned accounts received no refund.  
The mean of USD 14.42 is pulled up by a small number of larger refunds, with the maximum being USD 392.92.  
This confirms that refunds are the exception rather than the rule at RavenStack.

In [51]:
# categorical distributions and checks 

print("\n--- Churn Reason Codes ---")
print(churn_events['reason_code'].value_counts())

print("\n--- Preceding Upgrade Flag ---")
print(churn_events['preceding_upgrade_flag'].value_counts())

print("\n--- Preceding Downgrade Flag ---")
print(churn_events['preceding_downgrade_flag'].value_counts())

print("\n--- Reactivation Flag ---")
print(churn_events['is_reactivation'].value_counts())

print("\n--- Accounts in churn_events vs churned in accounts ---")
print(f"Unique account_ids in churn_events: {churn_events['account_id'].nunique()}")
print(f"Churned accounts in accounts table: {accounts['churn_flag'].sum()}")

print("\n--- Duplicate churn_event_id check ---")
print(f"Duplicate churn_event_ids: {churn_events['churn_event_id'].duplicated().sum()}") 



--- Churn Reason Codes ---
reason_code
features      114
support       104
budget        104
unknown        95
competitor     92
pricing        91
Name: count, dtype: int64

--- Preceding Upgrade Flag ---
preceding_upgrade_flag
False    477
True     123
Name: count, dtype: int64

--- Preceding Downgrade Flag ---
preceding_downgrade_flag
False    547
True      53
Name: count, dtype: int64

--- Reactivation Flag ---
is_reactivation
False    539
True      61
Name: count, dtype: int64

--- Accounts in churn_events vs churned in accounts ---
Unique account_ids in churn_events: 352
Churned accounts in accounts table: 110

--- Duplicate churn_event_id check ---
Duplicate churn_event_ids: 0


churn_events has 352 unique account_ids but accounts table only shows 110 churned accounts. Needs to be investigated.

In [52]:
# investigate churn discrepancy
print(f"Total rows in churn_events: {len(churn_events)}")
print(f"Unique accounts in churn_events: {churn_events['account_id'].nunique()}")
print(f"Accounts with churn_flag True in accounts: {accounts['churn_flag'].sum()}")

# check if churn_events accounts exist in accounts table
churn_ids = set(churn_events['account_id'].unique())
account_ids = set(accounts['account_id'].unique())

print(f"\nChurn account_ids found in accounts table: {len(churn_ids & account_ids)}")
print(f"Churn account_ids NOT in accounts table: {len(churn_ids - account_ids)}")

# check reactivations
print(f"\nReactivation events: {churn_events['is_reactivation'].sum()}")
print(f"Non-reactivation churn events: {(churn_events['is_reactivation'] == False).sum()}")


Total rows in churn_events: 600
Unique accounts in churn_events: 352
Accounts with churn_flag True in accounts: 110

Churn account_ids found in accounts table: 352
Churn account_ids NOT in accounts table: 0

Reactivation events: 61
Non-reactivation churn events: 539


**Distributions & checks:**  
Churn reasons are fairly evenly distributed - features (114) leads slightly, followed by support (104), budget (104), unknown (95), competitor (92), and pricing (91). No single reason dominates, suggesting churn is driven by multiple factors rather than one clear issue.  
123 churned accounts (20.5%) had an upgrade in the 90 days preceding churn (per dataset definition of preceding_upgrade_flag) - an unexpected pattern worth investigating in Q3.  
53 churned accounts (8.8%) had a preceding downgrade, which is a more expected churn signal.  
61 reactivation events recorded - accounts that previously churned and returned.  
A discrepancy exists between churn_events (352 unique accounts) and accounts.churn_flag (110 currently churned) - this is explained by accounts that churned multiple times or reactivated. churn_events is the historical log; accounts.churn_flag reflects current status only.  
No duplicate churn_event_ids - the table is clean.  